---
title: Find the Prime Minister and Ministry on a particular date
authors: 
    - Tim Sherratt
draft: true
---

You have a political speech, document, or event that you want to put into context, how do you find the Prime Minister or party that was in power on a particular date?

One way is by creating a dataset combining data from the Prime Ministers and Ministries endpoints of the [Australian Parliamentary Handbook API](/hansard/resources/parliamentary-handbook-api.ipynb). Then it's just a matter of querying the dataset by date.

In [1]:
import pandas as pd

In [27]:
# Get the Ministries data
df_min = pd.read_json(
    "https://handbookapi.aph.gov.au/api/StatisticalInformation/Ministries",
    convert_dates=["DateStart", "DateEnd"],
)

# Combine the parties data into a single field
df_min["parties_in_ministry"] = df_min.apply(
    lambda x: sorted([p["Name"] for p in x["Parties"]]), axis=1
)

# Reduce parties to a two-party category
df_min["party_type"] = df_min.apply(
    lambda x: (
        "Labor" if "Australian Labor Party" in x["parties_in_ministry"] else "Non-Labor"
    ),
    axis=1,
)

# Get PMs data
df_pm = pd.read_json(
    "https://handbookapi.aph.gov.au/api/HistoricalInformation/PrimeMinisters"
)[["Phid", "DisplayName"]]

# Merge Ministries and PMs
df = pd.merge(df_min, df_pm, how="left", left_on="LeaderPHID", right_on="Phid")[
    [
        "DisplayName",
        "MinistryName",
        "party_type",
        "parties_in_ministry",
        "DateStart",
        "DateEnd",
    ]
]

# Clean up column order and names
df.columns = [
    "prime_minister",
    "ministry",
    "party_type",
    "parties_in_ministry",
    "date_start",
    "date_end",
]

In [33]:
def find_gov_by_date(date):
    """
    Query the Ministries/PMs dataset by date.
    """
    govs = df.loc[(df["date_start"] <= date) & (df["date_end"] >= date)]
    if not govs.empty:
        return govs.to_dict(orient="records")

In [34]:
find_gov_by_date("1940-09-21")

[{'prime_minister': 'MENZIES, the Rt. Hon. Sir Robert Gordon, KT, CH, QC',
  'ministry': 'Menzies Ministry',
  'party_type': 'Non-Labor',
  'parties_in_ministry': ['Australian Country Party',
   'Liberal and Country Party',
   'United Australia Party'],
  'date_start': Timestamp('1939-04-26 00:00:00'),
  'date_end': Timestamp('1940-09-21 00:00:00')},
 {'prime_minister': 'MENZIES, the Rt. Hon. Sir Robert Gordon, KT, CH, QC',
  'ministry': 'Menzies Ministry (2nd)',
  'party_type': 'Non-Labor',
  'parties_in_ministry': ['Australian Country Party',
   'Liberal and Country Party',
   'United Australia Party'],
  'date_start': Timestamp('1940-09-21 00:00:00'),
  'date_end': Timestamp('1941-08-29 00:00:00')}]